In [ ]:
import os
import numpy as np
import pandas as pd
from keras.preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.models import Model
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score



In [ ]:
train_dir = 'MRNet-v1.0/train'
valid_dir = 'MRNet-v1.0/valid'
label_csv_files = ['MRNet-v1.0/train-acl.csv', 'MRNet-v1.0/valid-acl.csv']

base_model = VGG16(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

def load_npy_images(base_dir):
    images = []
    filenames = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.npy'):
                img = np.load(os.path.join(root, file))
                images.append(img)
                filenames.append(file)
    return images, filenames

def load_labels(label_csv_files):
    labels_dict = {}
    for label_file in label_csv_files:
        if os.path.exists(label_file):
            df = pd.read_csv(label_file, header=None, names=['index', 'label'])
            for idx, row in df.iterrows():
                filename = f"{str(int(row['index'])).zfill(4)}.npy"
                labels_dict[filename] = row['label']
        else:
            print(f"File {label_file} does not exist")
    return labels_dict

train_images, train_filenames = load_npy_images(train_dir)
valid_images, valid_filenames = load_npy_images(valid_dir)
labels_dict = load_labels(label_csv_files)

def preprocess_image(img_array):
    if len(img_array.shape) == 3:
        if img_array.shape[0] == 1:
            img_array = np.repeat(img_array, 3, axis=0)
        elif img_array.shape[0] > 3:
            img_array = img_array[:3]
        img_array = np.transpose(img_array, (1, 2, 0))
    img_array = image.array_to_img(img_array)
    img_array = img_array.resize((224, 224))
    img_array = image.img_to_array(img_array)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array

def extract_features(img_array, model):
    img_data = preprocess_image(img_array)
    features = model.predict(img_data)
    flat_features = features.flatten()
    return flat_features

features_list = []
labels_list = []
filenames_list = []

def process_images(images, filenames):
    for img, filename in zip(images, filenames):
        features = extract_features(img, model)
        features_list.append(features)
        label = labels_dict.get(filename, 'Unknown')
        labels_list.append(label)
        filenames_list.append(filename)
        print(f"Processed {filename}: label = {label}")

process_images(train_images, train_filenames)
process_images(valid_images, valid_filenames)

features_df = pd.DataFrame(features_list)
features_df.insert(0, 'filename', filenames_list)
features_df.insert(1, 'label', labels_list)

features_df.to_csv('extracted_feature_VGG16.csv', index=False)
print("Features and labels saved to extracted_feature_VGG16.csv")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from keras.applications.vgg16 import preprocess_input

def preprocess_image(img_array):
    if len(img_array.shape) == 3:
        if img_array.shape[0] == 1:
            img_array = np.repeat(img_array, 3, axis=0)
        elif img_array.shape[0] > 3:            img_array = img_array[:3]
        img_array = np.transpose(img_array, (1, 2, 0))
    img = Image.fromarray(np.uint8(img_array))
    img = img.resize((224, 224))    
    img_array = np.array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array

npy_file_path = 'MRNet-v1.0/train/axial/0001.npy'
npy_file_path2 = 'MRNet-v1.0/train/coronal/0001.npy'
npy_file_path3 = 'MRNet-v1.0/train/sagittal/0001.npy'
image_array = np.load(npy_file_path)
image_array2 = np.load(npy_file_path2)
image_array3 = np.load(npy_file_path3)

preprocessed_image = preprocess_image(image_array)
preprocessed_image2 = preprocess_image(image_array2)
preprocessed_image3 = preprocess_image(image_array3)

def show_image(img_array):
    img_array = np.squeeze(img_array, axis=0) 
    img_array = img_array + [103.939, 116.779, 123.68]
    img_array = img_array[..., ::-1]
    img_array = np.clip(img_array, 0, 255).astype('uint8') 
    plt.imshow(img_array)
    plt.title("Preprocessed Image for VGG16")
    plt.axis('off')
    plt.show()

show_image(preprocessed_image)
show_image(preprocessed_image2)
show_image(preprocessed_image3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from keras.applications.vgg16 import preprocess_input

def preprocess_image(img_array):
    if len(img_array.shape) == 3:
        if img_array.shape[0] == 1:
            img_array = np.repeat(img_array, 3, axis=0)
            print("Converted grayscale to RGB")
        elif img_array.shape[0] > 3:
            img_array = img_array[:3]
            print("Selected the first 3 channels for RGB conversion")
        img_array = np.transpose(img_array, (1, 2, 0))
    
    img = Image.fromarray(np.uint8(img_array))
    img = img.resize((224, 224))
    img_array_resized = np.array(img)
    
    img_array_preprocessed = np.expand_dims(img_array_resized, axis=0) 
    img_array_preprocessed = preprocess_input(img_array_preprocessed)
    
    return img_array_resized, img_array_preprocessed

npy_file_path = 'MRNet-v1.0/train/axial/0001.npy'
image_array = np.load(npy_file_path)

image_array_resized, preprocessed_image = preprocess_image(image_array)

def show_image(img_array, title):
    plt.imshow(img_array)
    plt.title(title)
    plt.axis('off')
    plt.show()

show_image(image_array_resized, "Image After RGB Conversion and Resizing")
show_image(preprocessed_image[0], "Image After VGG-16 Preprocessing")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def preprocess_image(img_array):
    if len(img_array.shape) == 3:
        if img_array.shape[0] == 1: 
            img_array = np.repeat(img_array, 3, axis=0)
            print("Converted grayscale to RGB")
        elif img_array.shape[0] > 3:  
            img_array = img_array[:3]
            print("Selected the first 3 channels for RGB conversion")
        img_array = np.transpose(img_array, (1, 2, 0)) 
    
    img = Image.fromarray(np.uint8(img_array))
    
    original_img = img.copy()
    
    img = img.resize((224, 224))
    resized_img = np.array(img)
    
    return original_img, resized_img

npy_file_path = 'MRNet-v1.0/train/axial/0001.npy'
image_array = np.load(npy_file_path)

original_img, resized_img = preprocess_image(image_array)

def show_image(img, title):
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()

show_image(np.array(original_img), "Original Image Before Resizing")

show_image(resized_img, "Image After Resizing to 224x224")


In [ ]:
features_df = pd.read_csv('extracted_feature_VGG16.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svm_model = svm.SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print(f"SVM Accuracy: {accuracy_svm}")
print(f"SVM Precision: {precision_svm}")
print(f"SVM Recall: {recall_svm}")
print(f"SVM F1 Score: {f1_svm}")

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
print("KNN Classification Report:")
print(classification_report(y_test, y_pred_knn))
print(f"KNN Accuracy: {accuracy_knn}")
print(f"KNN Precision: {precision_knn}")
print(f"KNN Recall: {recall_knn}")
print(f"KNN F1 Score: {f1_knn}")

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt, average='weighted')
recall_dt = recall_score(y_test, y_pred_dt, average='weighted')
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))
print(f"Decision Tree Accuracy: {accuracy_dt}")
print(f"Decision Tree Precision: {precision_dt}")
print(f"Decision Tree Recall: {recall_dt}")
print(f"Decision Tree F1 Score: {f1_dt}")

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

features_df = pd.read_csv('extracted_feature_VGG16.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=0.25, random_state=42)

X_train_combined = np.concatenate((X_train, X_valid), axis=0)
y_train_combined = np.concatenate((y_train, y_valid), axis=0)

svm_model = svm.SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm, average='weighted')
recall_svm = recall_score(y_test, y_pred_svm, average='weighted')
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))
print(f"SVM Accuracy: {accuracy_svm}")
print(f"SVM Precision: {precision_svm}")
print(f"SVM Recall: {recall_svm}")
print(f"SVM F1 Score: {f1_svm}")

knn_model = KNeighborsClassifier(n_neighbors=17)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

accuracy_knn = accuracy_score(y_test, y_pred_knn)
precision_knn = precision_score(y_test, y_pred_knn, average='weighted')
recall_knn = recall_score(y_test, y_pred_knn, average='weighted')
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
print("KNN Classification Report:")
print(classification_report(y_test, y_pred_knn))
print(f"KNN Accuracy: {accuracy_knn}")
print(f"KNN Precision: {precision_knn}")
print(f"KNN Recall: {recall_knn}")
print(f"KNN F1 Score: {f1_knn}")

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt, average='weighted')
recall_dt = recall_score(y_test, y_pred_dt, average='weighted')
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))
print(f"Decision Tree Accuracy: {accuracy_dt}")
print(f"Decision Tree Precision: {precision_dt}")
print(f"Decision Tree Recall: {recall_dt}")
print(f"Decision Tree F1 Score: {f1_dt}")


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

features_df = pd.read_csv('extracted_feature_VGG16.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

best_n_neighbors = 1
best_accuracy = 0.0

for n_neighbors in range(1, 501):
    print('Model n:', n_neighbors)
    knn_model = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn_model.fit(X_train, y_train)
    y_pred_knn = knn_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred_knn)
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_n_neighbors = n_neighbors

print(f"Best n_neighbors: {best_n_neighbors}")
print(f"Best Accuracy: {best_accuracy}")

best_knn_model = KNeighborsClassifier(n_neighbors=best_n_neighbors)
best_knn_model.fit(X_train, y_train)
y_pred_knn_best = best_knn_model.predict(X_test)
print("\nBest KNN Classification Report:")
print(classification_report(y_test, y_pred_knn_best))


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from tqdm import tqdm

print("Loading data...")
features_df = pd.read_csv('extracted_feature_VGG16.csv')

X = features_df.drop(columns=['filename', 'label']).values
y = features_df['label'].values

print("Splitting data into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X,  y, test_size=0.2, random_state=42)

svm_model = SVC(probability=True)
knn_model = KNeighborsClassifier(n_neighbors=5)
dt_model = DecisionTreeClassifier()

ensemble_model = VotingClassifier(
    estimators=[('svm', svm_model), ('knn', knn_model), ('dt', dt_model)],
    voting='soft'
)

print("Training ensemble model...")
ensemble_model.fit(X_train, y_train)

print("Making predictions on training data...")
y_pred_train_ensemble = ensemble_model.predict(X_train)

print("Making predictions on test data...")
y_pred_test_ensemble = ensemble_model.predict(X_test)

train_accuracy_ensemble = accuracy_score(y_train, y_pred_train_ensemble)
test_accuracy_ensemble = accuracy_score(y_test, y_pred_test_ensemble)
test_recall_ensemble = recall_score(y_test, y_pred_test_ensemble, average='weighted')
test_f1_ensemble = f1_score(y_test, y_pred_test_ensemble, average='weighted')

print("\nEnsemble Model (VotingClassifier) Classification Report:")
print(classification_report(y_test, y_pred_test_ensemble))
print(f"Ensemble Training Accuracy: {train_accuracy_ensemble}")
print(f"Ensemble Test Accuracy: {test_accuracy_ensemble}")
print(f"Ensemble Test Recall: {test_recall_ensemble}")
print(f"Ensemble Test F1 Score: {test_f1_ensemble}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

npy_file_path = 'MRNet-v1.0/train/sagittal/0001.npy'
npy_file_path2 = 'MRNet-v1.0/train/axial/0001.npy'
npy_file_path3 = 'MRNet-v1.0/train/coronal/0001.npy'
images = np.load(npy_file_path)
images2 = np.load(npy_file_path2)
images3 = np.load(npy_file_path3)

def display_images(images, num_images=2):
    plt.figure(figsize=(10, 5))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i], cmap='gray')
        plt.title(f'Image {i+1}')
        plt.axis('off')
    plt.show()

display_images(images)
display_images(images2)
display_images(images3)
